<h1 style="font-family: 'Times New Roman'; text-align: center;"><b>Machine Learning</b></h1>

<h2 style="font-family: 'Times New Roman'; text-align: center;"><b>URL Phishing Detection Project</b></h2>

<h3 style="font-family: 'Times New Roman'; text-align: right;"><b>- Pallav Biyala (24110234)</b></h3>

<hr>

<h2 style="font-family: 'Times New Roman'; text-align: center;"><b> Model Comparision & Testing Time</b></h2>

### Importing Libraries:

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time
import math
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from sklearn.exceptions import ConvergenceWarning
import uuid, os, json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Logistic regression & XGBoost 
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline # this performs regularization
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
## Performance metrics
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    confusion_matrix,
    balanced_accuracy_score,
    precision_score,
    classification_report
)

In [2]:
df = pd.read_csv("../data/PhiUSIIL_Phishing_URL_Dataset.csv")

<hr>

### Adding our Datasets:

In [3]:
# Remove repeated URLs before splitting.
df = df.drop_duplicates(subset=["URL"], keep="first").copy()

# All 50 numeric input features, excluding the target.
all_numeric_features = [
    c for c in df.select_dtypes(include=np.number).columns
    if c != "label"
]

print("Total numeric features:", len(all_numeric_features))

# M1: Strict lexical URL features.
lexical_features = [
    "URLLength", "DomainLength", "TLDLength", "NoOfSubDomain",
    "NoOfLettersInURL", "LetterRatioInURL",
    "NoOfDegitsInURL", "DegitRatioInURL",
    "NoOfEqualsInURL", "NoOfQMarkInURL",
    "NoOfAmpersandInURL", "NoOfOtherSpecialCharsInURL",
    "SpacialCharRatioInURL", "IsDomainIP",
    "HasObfuscation", "NoOfObfuscatedChar", "ObfuscationRatio"
]

# M2: Add the three derived statistical features.
derived_features = [
    "TLDLegitimateProb",
    "URLCharProb",
    "CharContinuationRate"
]
# M3 adds HTTPS along with lexial and derived features --> represents what we can have if we are just given URL
# M4: Add HTTPS and the page-dependent features.
# URLSimilarityIndex is deliberately reserved for M5.
page_features = [
    c for c in all_numeric_features
    if c not in lexical_features
    and c not in derived_features
    and c not in ["IsHTTPS", "URLSimilarityIndex"]
]

feature_sets = {
    "M1 - Lexical": lexical_features,

    "M2 - Derived": (
        lexical_features + derived_features
    ),

    "M3 - URL Features": (
        lexical_features + derived_features + ["IsHTTPS"]
    ),

    "M4 - Page-aware": (
        lexical_features
        + derived_features
        + ["IsHTTPS"]
        + page_features
    ),

    "M5 - All numeric": all_numeric_features
}

for name, features in feature_sets.items():
    missing = set(features) - set(df.columns)
    assert not missing, f"{name}: missing columns {missing}"
    assert len(features) == len(set(features)), (
        f"{name}: duplicate feature names"
    )
    print(f"{name}: {len(features)} features")

assert "URLSimilarityIndex" not in feature_sets["M4 - Page-aware"]
assert "URLSimilarityIndex" in feature_sets["M5 - All numeric"]

Total numeric features: 50
M1 - Lexical: 17 features
M2 - Derived: 20 features
M3 - URL Features: 21 features
M4 - Page-aware: 49 features
M5 - All numeric: 50 features


In [4]:
X = df[all_numeric_features]
y = df["label"]

# 80% train+validation, 10% test.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.10,
    random_state=42,
    stratify=y
)

print("Train:", len(X_train_val))
print("Test:", len(X_test))

Train: 211833
Test: 23537


<hr>

### Loading Data from JSON file:

In [6]:
RESULTS_DIR = "../results"

def load_results(folder):
    filepath = os.path.join(
        RESULTS_DIR,
        folder,
        "all_results.json"
    )

    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

#### Loading Baseline Models

In [7]:
baseline_results = load_results("Baseline Models")

print("Baseline experiments:", len(baseline_results))

Baseline experiments: 55


In [8]:
baseline_df = pd.DataFrame(baseline_results)

display(
    baseline_df[
        [
            "model",
            "feature_set",
            "validation_accuracy",
            "phishing_recall",
            "missed_phishing",
            "false_alarms",
            "training_time_seconds"
        ]
    ]
)

,model,feature_set,validation_accuracy,phishing_recall,missed_phishing,false_alarms,training_time_seconds
0,Majority Class,M1 - Lexical,0.572928,0.000000,10052,0,0.007306
1,Majority Class,M2 - Derived,0.572928,0.000000,10052,0,0.007112
2,Majority Class,M3 - URL Features,0.572928,0.000000,10052,0,0.006995
3,Majority Class,M4 - Page-aware,0.572928,0.000000,10052,0,0.007288
4,Majority Class,M5 - All numeric,0.572928,0.000000,10052,0,0.006082
5,Logistic Regression,M1 - Lexical,0.997408,0.994429,56,5,1.157367
6,Logistic Regression,M2 - Derived,0.997408,0.994329,57,4,1.677233
7,Logistic Regression,M3 - URL Features,0.997323,0.994429,56,7,0.700146
8,Logistic Regression,M4 - Page-aware,0.999575,0.999503,5,5,0.667193
9,Logistic Regression,M5 - All numeric,1.000000,1.000000,0,0,0.556216


#### Loading Tuned Models:

In [9]:
RESULTS_DIR = Path("../results")

tuning_results = []

for filepath in RESULTS_DIR.rglob("all_results.json"):

    # Skip baseline experiments
    if filepath.parent.name == "Baseline Models":
        continue

    with open(filepath, "r", encoding="utf-8") as f:
        results = json.load(f)

    print(f"Loaded {len(results)} experiments from: {filepath.parent.name}")

    tuning_results.extend(results)

tuning_df = pd.DataFrame(tuning_results)

print("\nTotal tuning experiments:", len(tuning_df))
print("Unique models:", tuning_df["model"].nunique())
print("Feature sets:", tuning_df["feature_set"].nunique())

Loaded 45 experiments from: Ada Boosting
Loaded 300 experiments from: Decision Trees
Loaded 135 experiments from: Gradient Boosting
Loaded 35 experiments from: Linear SVM
Loaded 90 experiments from: Random Forests
Loaded 35 experiments from: Ridge Logistic Regression
Loaded 135 experiments from: XG Boosting

Total tuning experiments: 775
Unique models: 7
Feature sets: 5


In [10]:
print("DataFrame shape:", tuning_df.shape)
print("\nColumns:")
print(tuning_df.columns.tolist())

DataFrame shape: (775, 16)

Columns:
['experiment_id', 'model', 'experiment_type', 'feature_set', 'n_features', 'hyperparameters', 'training_time_seconds', 'train_accuracy', 'validation_accuracy', 'balanced_accuracy', 'phishing_recall', 'missed_phishing', 'false_alarms', 'evaluation_split', 'converged', 'convergence_warning_messages']


In [11]:
## this shows how many experiments each model ran on each feature set
display(
    tuning_df.groupby(["model", "feature_set"])
    .size()
    .unstack(fill_value=0)
)

feature_set,M1 - Lexical,M2 - Derived,M3 - URL Features,M4 - Page-aware,M5 - All numeric
model,,,,,
AdaBoost,9,9,9,9,9
Decision Tree,60,60,60,60,60
Gradient Boosting,27,27,27,27,27
Linear SVM,7,7,7,7,7
Random Forest,18,18,18,18,18
Ridge Logistic Regression,7,7,7,7,7
XGBoost,27,27,27,27,27


<hr>

### Combining all the Experiments:

In [12]:
all_experiments_df = pd.concat(
    [baseline_df, tuning_df],
    ignore_index=True
)

print("Total experiments:", len(all_experiments_df))
print("Baseline experiments:", len(baseline_df))
print("Tuning experiments:", len(tuning_df))

assert len(all_experiments_df) == 830

print("\nAll 830 experiments combined successfully!")

Total experiments: 830
Baseline experiments: 55
Tuning experiments: 775

All 830 experiments combined successfully!


In [13]:
# Inspect available models
print("Unique models:")
print(sorted(all_experiments_df["model"].unique()))

print("\nFeature sets:")
print(sorted(all_experiments_df["feature_set"].unique()))

print("\nExperiment counts by model and feature set:")
display(
    all_experiments_df.groupby(["model", "feature_set"])
    .size()
    .unstack(fill_value=0)
)

Unique models:
['AdaBoost', 'Decision Tree', 'Gradient Boosting', 'Lasso Logistic Regression', 'Linear SVM', 'Logistic Regression', 'Majority Class', 'RBF SVM', 'Random Forest', 'Ridge Logistic Regression', 'XGBoost']

Feature sets:
['M1 - Lexical', 'M2 - Derived', 'M3 - URL Features', 'M4 - Page-aware', 'M5 - All numeric']

Experiment counts by model and feature set:


feature_set,M1 - Lexical,M2 - Derived,M3 - URL Features,M4 - Page-aware,M5 - All numeric
model,,,,,
AdaBoost,10,10,10,10,10
Decision Tree,61,61,61,61,61
Gradient Boosting,28,28,28,28,28
Lasso Logistic Regression,1,1,1,1,1
Linear SVM,8,8,8,8,8
Logistic Regression,1,1,1,1,1
Majority Class,1,1,1,1,1
RBF SVM,1,1,1,1,1
Random Forest,19,19,19,19,19


<hr>

### Model Comparision

In [14]:
# ============================================================
# FINAL MODEL SELECTION — VALIDATION RESULTS
# ============================================================

retained_models = [
    "Logistic Regression",
    "Ridge Logistic Regression",
    "Linear SVM",
    "Decision Tree",
    "Random Forest",
    "Gradient Boosting",
    "AdaBoost",
    "XGBoost",
    "Majority Class"
]

feature_sets = [
    "M1 - Lexical",
    "M2 - Derived",
    "M3 - URL Features",
    "M4 - Page-aware",
    "M5 - All numeric"
]

# Keep only our selected models
selection_df = all_experiments_df[
    all_experiments_df["model"].isin(retained_models)
].copy()

# Keep validation experiments only
selection_df = selection_df[
    selection_df["evaluation_split"] == "validation"
].copy()

print("Experiments available for selection:", len(selection_df))
print("Models retained:", selection_df["model"].nunique())
print("Feature sets:", selection_df["feature_set"].nunique())

Experiments available for selection: 820
Models retained: 9
Feature sets: 5


In [15]:
# ============================================================
# SELECT BEST CONFIGURATION FOR EACH MODEL-FEATURE SET
# ============================================================

selected_rows = []

for model_name in retained_models:

    for feature_set in feature_sets:

        candidates = selection_df[
            (selection_df["model"] == model_name) &
            (selection_df["feature_set"] == feature_set)
        ].copy()

        if candidates.empty:
            raise ValueError(
                f"No experiments found for {model_name} | {feature_set}"
            )

        # Exclude experiments that did not converge
        candidates = candidates[
            candidates["converged"] == True
        ]

        if candidates.empty:
            raise ValueError(
                f"No converged experiments for {model_name} | {feature_set}"
            )

        # Ranking priority:
        # 1. Highest phishing recall
        # 2. Highest balanced accuracy
        # 3. Highest validation accuracy
        # 4. Fewest false alarms
        # 5. Lowest training time

        candidates = candidates.sort_values(
            by=[
                "phishing_recall",
                "balanced_accuracy",
                "validation_accuracy",
                "false_alarms",
                "training_time_seconds"
            ],
            ascending=[
                False,
                False,
                False,
                True,
                True
            ],
            kind="stable"
        )

        best = candidates.iloc[0].copy()
        selected_rows.append(best)

# Create final selection DataFrame
best_configs_df = pd.DataFrame(selected_rows).reset_index(drop=True)

print("Selected configurations:", len(best_configs_df))

display(
    best_configs_df[
        [
            "model",
            "feature_set",
            "experiment_type",
            "hyperparameters",
            "phishing_recall",
            "balanced_accuracy",
            "validation_accuracy",
            "missed_phishing",
            "false_alarms",
            "training_time_seconds"
        ]
    ]
)

Selected configurations: 45


,model,feature_set,experiment_type,hyperparameters,phishing_recall,balanced_accuracy,validation_accuracy,missed_phishing,false_alarms,training_time_seconds
0,Logistic Regression,M1 - Lexical,Baseline,"{'C': 1.0, 'class_weight': None, 'dual': False...",0.994429,0.997029,0.997408,56,5,1.157367
1,Logistic Regression,M2 - Derived,Baseline,"{'C': 1.0, 'class_weight': None, 'dual': False...",0.994329,0.997016,0.997408,57,4,1.677233
2,Logistic Regression,M3 - URL Features,Baseline,"{'C': 1.0, 'class_weight': None, 'dual': False...",0.994429,0.996955,0.997323,56,7,0.700146
3,Logistic Regression,M4 - Page-aware,Baseline,"{'C': 1.0, 'class_weight': None, 'dual': False...",0.999503,0.999566,0.999575,5,5,0.667193
4,Logistic Regression,M5 - All numeric,Baseline,"{'C': 1.0, 'class_weight': None, 'dual': False...",1.000000,1.000000,1.000000,0,0,0.556216
5,Ridge Logistic Regression,M1 - Lexical,Hyperparameter Tuning,"{'penalty': 'l2', 'C': 100.0, 'solver': 'lbfgs...",0.994230,0.997115,0.997536,58,0,1.178998
6,Ridge Logistic Regression,M2 - Derived,Hyperparameter Tuning,"{'penalty': 'l2', 'C': 1000.0, 'solver': 'lbfg...",0.994031,0.997016,0.997451,60,0,2.018717
7,Ridge Logistic Regression,M3 - URL Features,Hyperparameter Tuning,"{'penalty': 'l2', 'C': 1000.0, 'solver': 'lbfg...",0.994429,0.996992,0.997366,56,6,0.842372
8,Ridge Logistic Regression,M4 - Page-aware,Hyperparameter Tuning,"{'penalty': 'l2', 'C': 1000.0, 'solver': 'lbfg...",0.999503,0.999566,0.999575,5,5,0.642326
9,Ridge Logistic Regression,M5 - All numeric,Hyperparameter Tuning,"{'penalty': 'l2', 'C': 100.0, 'solver': 'lbfgs...",1.000000,1.000000,1.000000,0,0,0.489751


In [16]:
# ============================================================
# VERIFY MODEL SELECTION
# ============================================================

assert len(best_configs_df) == 45

assert best_configs_df.groupby(
    ["model", "feature_set"]
).size().eq(1).all()

assert set(best_configs_df["model"]) == set(retained_models)

assert set(best_configs_df["feature_set"]) == set(feature_sets)

print("SUCCESS: Exactly 45 configurations selected!")
print("Each model-feature-set combination has exactly one winner.")

display(
    best_configs_df.pivot(
        index="model",
        columns="feature_set",
        values="experiment_id"
    )
)

SUCCESS: Exactly 45 configurations selected!
Each model-feature-set combination has exactly one winner.


feature_set,M1 - Lexical,M2 - Derived,M3 - URL Features,M4 - Page-aware,M5 - All numeric
model,,,,,
AdaBoost,f03c737f-575c-40da-b01e-73402481fbc5,2ecf72c2-c49a-4262-bd83-ce03cb20f964,1bbb4273-c9d7-4a53-8259-0f41f392b093,03c53768-f14c-4b88-a434-3fcc2c526f5e,2fee65b6-170b-4423-a3e1-267a9520214d
Decision Tree,6bec0528-c448-4846-9f14-0d41a93b4017,bf2e82b9-4edf-4f1d-b92e-05ddfec1b0db,601b9ad6-c8e8-46ca-9857-2fe44db443fe,24c34407-1890-4fb1-a2a4-d4651c47500b,888a5709-1821-4d8a-8972-9123bc9d1512
Gradient Boosting,c1268fae-9225-4a30-b463-54fc66e6f9bd,7d78875a-7d7c-4baa-9db1-aefd7694ba54,56d7692d-8d9a-475b-8e47-c80eaa069007,b96dbc66-7cf2-43ef-9035-accbed44981d,2486dfa8-03be-4def-8651-79a3336265c8
Linear SVM,9fa64dc5-ad4a-40a5-a818-751a18babed9,4a292fed-3514-4d25-97ae-77d2f1cf13d4,2058e99e-0e7c-4f3b-8056-dccfde4149ff,49f7a33e-118e-44e9-ab13-6a86cf89a4c5,7ff6f646-cbdc-4794-989e-88e0d9678d08
Logistic Regression,f47a40dc-e81a-4787-a84a-18f398bf29ff,f5105ea3-b411-4a84-8369-e342c81c3fac,878c9f21-6c06-443f-8202-b9a3ee6d3ad9,0bd736f6-a5bf-453a-a6d9-a8d62a7ddfd1,fbdf31b8-2d2e-413c-b38b-76c12d1db940
Majority Class,7782f40a-8280-487c-9dcd-220d395c47b0,4fa35811-a497-41e4-ba57-823bf6bfa751,71719cc1-b20d-4df8-9e31-09a919421208,78219e5f-139f-44c1-b12a-8fabcd7e31fe,9831a2d1-f6d7-4d46-bc99-c57b3e0f3451
Random Forest,e057771f-f2f1-4c86-a2e8-f819b91d7838,f7e3366c-92f8-49eb-bdab-ff04ed2f5c70,7e253397-6e9e-4749-ab7a-27d672fc24ba,8d58a72a-4127-400f-9475-323a15f4dea4,d6d8a25d-0330-4b02-a106-0cacbe257f54
Ridge Logistic Regression,ade20aaa-6954-4b27-beef-e99f367bd60b,f3d11542-b132-4dc9-bcd2-7f8f0ebb1b18,5013c54d-73c3-45b4-a706-393a6d371dd4,a51cb424-7f64-4860-b920-888a843ae313,3bc24fbe-d633-4483-a080-70aae95e8557
XGBoost,7ee3929e-01a7-41bd-8aff-3a654cf2162a,2a4bd484-a00c-47e4-b28a-b9911698d784,374f4e22-d376-48cd-aa92-f8a9bc29de85,1fcbe245-7069-4c49-89e7-3e1d5719d676,cbd5fe34-9fdd-440b-8517-d194982f150d


In [17]:
for model_name in [
    "Logistic Regression",
    "Ridge Logistic Regression"
]:
    print(f"\n{model_name}")

    row = best_configs_df[
        best_configs_df["model"] == model_name
    ].iloc[0]

    print(row["hyperparameters"])


Logistic Regression
{'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': None, 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}

Ridge Logistic Regression
{'penalty': 'l2', 'C': 100.0, 'solver': 'lbfgs', 'max_iter': 1000}


In [18]:
# ============================================================
# INSPECT ALL SELECTED HYPERPARAMETERS
# ============================================================

for model_name in retained_models:

    print(f"\n{'=' * 80}")
    print(model_name)
    print("=" * 80)

    model_rows = best_configs_df[
        best_configs_df["model"] == model_name
    ]

    for _, row in model_rows.iterrows():

        print(f"\n{row['feature_set']}")
        print("Experiment:", row["experiment_type"])
        print("Hyperparameters:", row["hyperparameters"])


Logistic Regression

M1 - Lexical
Experiment: Baseline
Hyperparameters: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': None, 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}

M2 - Derived
Experiment: Baseline
Hyperparameters: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': None, 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}

M3 - URL Features
Experiment: Baseline
Hyperparameters: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': None, 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}

M4 - Page-aware
Experiment: Baseline
Hyperparameters: {'C': 1

<hr>

### Testing Time!

In [19]:
feature_columns = {
    "M1 - Lexical": lexical_features,

    "M2 - Derived": (
        lexical_features + derived_features
    ),

    "M3 - URL Features": (
        lexical_features + derived_features + ["IsHTTPS"]
    ),

    "M4 - Page-aware": (
        lexical_features
        + derived_features
        + ["IsHTTPS"]
        + page_features
    ),

    "M5 - All numeric": all_numeric_features
}

assert len(feature_columns) == 5

for name, features in feature_columns.items():
    print(f"{name}: {len(features)} features")

M1 - Lexical: 17 features
M2 - Derived: 20 features
M3 - URL Features: 21 features
M4 - Page-aware: 49 features
M5 - All numeric: 50 features


In [20]:
# ============================================================
# FINAL MODEL FACTORY
# ============================================================

def create_final_model(row):

    model_name = row["model"]
    params = row["hyperparameters"]

    # --------------------------------------------------------
    # 1. Logistic Regression (Unregularized)
    # --------------------------------------------------------
    if model_name == "Logistic Regression":

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                penalty=None,
                solver="lbfgs",
                max_iter=1000,
                random_state=42
            ))
        ])

    # --------------------------------------------------------
    # 2. Ridge Logistic Regression (L2)
    # --------------------------------------------------------
    elif model_name == "Ridge Logistic Regression":

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                l1_ratio=0,
                C=params["C"],
                solver="lbfgs",
                max_iter=1000,
                tol=1e-4,
                random_state=42
            ))
        ])

    # --------------------------------------------------------
    # 3. Linear SVM
    # --------------------------------------------------------
    elif model_name == "Linear SVM":

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LinearSVC(
                C=params["C"],
                max_iter=3000,
                random_state=42
            ))
        ])

    # --------------------------------------------------------
    # 4. Decision Tree
    # --------------------------------------------------------
    elif model_name == "Decision Tree":

        model = DecisionTreeClassifier(
            criterion=params["criterion"],
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42
        )

    # --------------------------------------------------------
    # 5. Random Forest
    # --------------------------------------------------------
    elif model_name == "Random Forest":

        model = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            criterion="gini",
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        )

    # --------------------------------------------------------
    # 6. Gradient Boosting
    # --------------------------------------------------------
    elif model_name == "Gradient Boosting":

        model = GradientBoostingClassifier(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            random_state=42
        )

    # --------------------------------------------------------
    # 7. AdaBoost
    # --------------------------------------------------------
    elif model_name == "AdaBoost":

        model = AdaBoostClassifier(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            random_state=42
        )

    # --------------------------------------------------------
    # 8. XGBoost
    # --------------------------------------------------------
    elif model_name == "XGBoost":

        model = XGBClassifier(
            objective="binary:logistic",
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            subsample=1.0,
            random_state=42,
            n_jobs=-1,
            eval_metric="logloss"
        )

    # --------------------------------------------------------
    # 9. Majority Class
    # --------------------------------------------------------
    elif model_name == "Majority Class":

        model = DummyClassifier(
            strategy="most_frequent"
        )

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model

In [21]:
# ============================================================
# RECREATE ALL 45 SELECTED MODELS
# ============================================================

final_models = {}

for _, row in best_configs_df.iterrows():

    model_name = row["model"]
    feature_set = row["feature_set"]

    model = create_final_model(row)

    final_models[(model_name, feature_set)] = model

print("Total recreated models:", len(final_models))

assert len(final_models) == 45

print("\nAll 45 selected models recreated successfully!")

Total recreated models: 45

All 45 selected models recreated successfully!


In [22]:
# ============================================================
# VERIFY FINAL MODEL COLLECTION
# ============================================================

for model_name in retained_models:

    print(f"\n{'=' * 65}")
    print(model_name)
    print("=" * 65)

    for feature_set in feature_sets:

        model = final_models[(model_name, feature_set)]

        print(
            f"{feature_set:<25} -> "
            f"{type(model).__name__}"
        )

assert len(final_models) == 9 * 5

print("\nSUCCESS: All 45 estimators are ready for final training.")


Logistic Regression
M1 - Lexical              -> Pipeline
M2 - Derived              -> Pipeline
M3 - URL Features         -> Pipeline
M4 - Page-aware           -> Pipeline
M5 - All numeric          -> Pipeline

Ridge Logistic Regression
M1 - Lexical              -> Pipeline
M2 - Derived              -> Pipeline
M3 - URL Features         -> Pipeline
M4 - Page-aware           -> Pipeline
M5 - All numeric          -> Pipeline

Linear SVM
M1 - Lexical              -> Pipeline
M2 - Derived              -> Pipeline
M3 - URL Features         -> Pipeline
M4 - Page-aware           -> Pipeline
M5 - All numeric          -> Pipeline

Decision Tree
M1 - Lexical              -> DecisionTreeClassifier
M2 - Derived              -> DecisionTreeClassifier
M3 - URL Features         -> DecisionTreeClassifier
M4 - Page-aware           -> DecisionTreeClassifier
M5 - All numeric          -> DecisionTreeClassifier

Random Forest
M1 - Lexical              -> RandomForestClassifier
M2 - Derived              ->

In [48]:
# ============================================================
# FINAL TRAINING AND TESTING — 45 CONFIGURATIONS
# ============================================================

test_results = []

for index, (_, row) in enumerate(best_configs_df.iterrows(), start=1):

    model_name = row["model"]
    feature_set = row["feature_set"]

    print(f"\n[{index}/45] {model_name} | {feature_set}")

    # Retrieve the recreated estimator.
    model = final_models[(model_name, feature_set)]

    # Retrieve the corresponding feature columns.
    features = feature_columns[feature_set]

    # Prepare training and test data.
    X_train_selected = X_train_val[features]
    X_test_selected = X_test[features]

    # --------------------------------------------------------
    # TRAIN ON THE COMPLETE 90% TRAINING DATA
    # --------------------------------------------------------

    start_time = time.time()

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")

        model.fit(X_train_selected, y_train_val)

    training_time = time.time() - start_time

    # Capture convergence warnings.
    convergence_warnings = [
        w for w in caught
        if issubclass(w.category, ConvergenceWarning)
    ]

    converged = len(convergence_warnings) == 0

    warning_messages = [
        str(w.message) for w in convergence_warnings
    ]

    # --------------------------------------------------------
    # PREDICT ON THE UNTOUCHED 10% TEST DATA
    # --------------------------------------------------------

    y_test_pred = model.predict(X_test_selected)

    # --------------------------------------------------------
    # CALCULATE PERFORMANCE METRICS
    # --------------------------------------------------------

    test_accuracy = accuracy_score(
        y_test, y_test_pred
    )

    test_balanced_accuracy = balanced_accuracy_score(
        y_test, y_test_pred
    )

    test_phishing_recall = recall_score(
        y_test, y_test_pred, pos_label=0
    )

    test_phishing_precision = precision_score(
        y_test, y_test_pred,
        pos_label=0,
        zero_division=0
    )

    # Confusion matrix: Phishing (0), Legitimate (1).
    cm = confusion_matrix(
        y_test,
        y_test_pred,
        labels=[0, 1]
    )

    missed_phishing = int(cm[0, 1])
    false_alarms = int(cm[1, 0])

    # --------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------

    result = {
        "experiment_id": str(uuid.uuid4()),

        "model": model_name,
        "feature_set": feature_set,
        "n_features": len(features),

        "experiment_type": row["experiment_type"],
        "hyperparameters": row["hyperparameters"],
        "selected_experiment_id": row["experiment_id"],

        "evaluation_split": "test",
        "refit_on": "train+validation (90%)",

        "training_time_seconds": training_time,
        "converged": converged,
        "convergence_warning_messages": warning_messages,

        "test_accuracy": test_accuracy,
        "test_balanced_accuracy": test_balanced_accuracy,
        "test_phishing_recall": test_phishing_recall,
        "test_phishing_precision": test_phishing_precision,

        "test_missed_phishing": missed_phishing,
        "test_false_alarms": false_alarms,

        "test_confusion_matrix": cm.tolist(),
        "confusion_matrix_label_order": [
            "Phishing (0)",
            "Legitimate (1)"
        ]
    }

    test_results.append(result)

    # --------------------------------------------------------
    # DISPLAY PROGRESS
    # --------------------------------------------------------

    print(
        f"Accuracy: {test_accuracy:.5f} | "
        f"Phishing Recall: {test_phishing_recall:.5f} | "
        f"Balanced Accuracy: {test_balanced_accuracy:.5f} | "
        f"Missed Phishing: {missed_phishing} | "
        f"False Alarms: {false_alarms} | "
        f"Time: {training_time:.2f}s"
    )


# ============================================================
# FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 75)
print("FINAL TESTING COMPLETED")
print("=" * 75)

print("Total evaluations:", len(test_results))
print("Converged:", sum(r["converged"] for r in test_results))
print("Non-converged:", sum(not r["converged"] for r in test_results))

assert len(test_results) == 45

assert len({
    (r["model"], r["feature_set"])
    for r in test_results
}) == 45

print("\nSUCCESS: All 45 final evaluations completed!")


[1/45] Logistic Regression | M1 - Lexical
Accuracy: 0.99737 | Phishing Recall: 0.99413 | Balanced Accuracy: 0.99695 | Missed Phishing: 59 | False Alarms: 3 | Time: 1.47s

[2/45] Logistic Regression | M2 - Derived
Accuracy: 0.99737 | Phishing Recall: 0.99413 | Balanced Accuracy: 0.99695 | Missed Phishing: 59 | False Alarms: 3 | Time: 1.59s

[3/45] Logistic Regression | M3 - URL Features
Accuracy: 0.99703 | Phishing Recall: 0.99373 | Balanced Accuracy: 0.99661 | Missed Phishing: 63 | False Alarms: 7 | Time: 0.88s

[4/45] Logistic Regression | M4 - Page-aware
Accuracy: 0.99936 | Phishing Recall: 0.99901 | Balanced Accuracy: 0.99932 | Missed Phishing: 10 | False Alarms: 5 | Time: 0.89s

[5/45] Logistic Regression | M5 - All numeric
Accuracy: 0.99992 | Phishing Recall: 0.99980 | Balanced Accuracy: 0.99990 | Missed Phishing: 2 | False Alarms: 0 | Time: 0.62s

[6/45] Ridge Logistic Regression | M1 - Lexical
Accuracy: 0.99737 | Phishing Recall: 0.99393 | Balanced Accuracy: 0.99693 | Missed Ph

In [49]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Dedicated directory for final test evaluations
final_dir = Path("results/Final_Tests")
final_dir.mkdir(parents=True, exist_ok=True)

# Separate archive: preserve all 45 detailed evaluations
json_path = final_dir / "final_45_test_results.json"

# Avoid accidentally overwriting an existing experiment archive
if json_path.exists():
    raise FileExistsError(
        f"{json_path} already exists. Choose another filename to preserve it."
    )

# Convert NumPy objects into JSON-compatible Python objects
def json_serializer(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, np.bool_):
        return bool(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        test_results,
        f,
        indent=4,
        default=json_serializer
    )

print(f"Saved {len(test_results)} final evaluations to: {json_path}")

Saved 45 final evaluations to: results\final\final_45_test_results.json


In [50]:
csv_path = final_dir / "final_45_test_summary.csv"

if csv_path.exists():
    raise FileExistsError(f"{csv_path} already exists.")

test_summary_df = pd.DataFrame(test_results)

# Exclude nested confusion matrices from the CSV
test_summary_df = test_summary_df.drop(
    columns=["test_confusion_matrix"],
    errors="ignore"
)

test_summary_df.to_csv(csv_path, index=False)

print(f"Saved CSV summary to: {csv_path}")
print(f"Rows: {len(test_summary_df)}")

Saved CSV summary to: results\final\final_45_test_summary.csv
Rows: 45


In [51]:
with open(json_path, "r", encoding="utf-8") as f:
    saved_results = json.load(f)

assert len(saved_results) == 45

assert all(
    result["evaluation_split"] == "test"
    for result in saved_results
)

assert all(
    result["converged"]
    for result in saved_results
)

print("SUCCESS: Final test archive verified!")
print(f"Evaluations: {len(saved_results)}")
print(f"Models: {len(set(r['model'] for r in saved_results))}")
print(f"Feature sets: {len(set(r['feature_set'] for r in saved_results))}")

SUCCESS: Final test archive verified!
Evaluations: 45
Models: 9
Feature sets: 5


In [52]:
import json
from pathlib import Path

final_dir = Path("results/Final_Tests")
confusion_path = final_dir / "final_45_test_confusion_matrices.json"

if confusion_path.exists():
    raise FileExistsError(
        f"{confusion_path} already exists. Not overwriting it."
    )

final_confusion_matrices = {}

for result in test_results:
    key = f"{result['model']} | {result['feature_set']}"

    final_confusion_matrices[key] = {
        "model": result["model"],
        "feature_set": result["feature_set"],
        "confusion_matrix": result["test_confusion_matrix"],
        "labels": result["confusion_matrix_label_order"]
    }

with open(confusion_path, "w", encoding="utf-8") as f:
    json.dump(final_confusion_matrices, f, indent=4)

print(f"Saved {len(final_confusion_matrices)} confusion matrices!")
print(f"Path: {confusion_path}")

Saved 45 confusion matrices!
Path: results\final\final_45_test_confusion_matrices.json


In [53]:
with open(confusion_path, "r", encoding="utf-8") as f:
    saved_matrices = json.load(f)

assert len(saved_matrices) == 45

print("SUCCESS: Final confusion matrix archive verified!")
print("Total matrices:", len(saved_matrices))

SUCCESS: Final confusion matrix archive verified!
Total matrices: 45


### Building Data to comapre Feature Sets

In [15]:
# ============================================================
# FEATURE SET COMPOSITION
# ============================================================

for name, features in feature_columns.items():

    print(f"\n{'=' * 65}")
    print(name)
    print("=" * 65)

    print("Number of features:", len(features))

    print(features)


M1 - Lexical
Number of features: 17
['URLLength', 'DomainLength', 'TLDLength', 'NoOfSubDomain', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsDomainIP', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio']

M2 - Derived
Number of features: 20
['URLLength', 'DomainLength', 'TLDLength', 'NoOfSubDomain', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsDomainIP', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'TLDLegitimateProb', 'URLCharProb', 'CharContinuationRate']

M3 - URL Features
Number of features: 21
['URLLength', 'DomainLength', 'TLDLength', 'NoOfSubDomain', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL

In [12]:
# ============================================================
# FEATURE SET ANALYSIS — EXCLUDING MAJORITY BASELINE
# ============================================================

learning_models_df = test_results_df[
    test_results_df["model"] != "Majority Class"
].copy()

learning_model_stats = (
    learning_models_df
    .groupby("feature_set", observed=False)[performance_metrics]
    .agg(["min", "max", "mean", "median"])
    * 100
)

print("Models included:", learning_models_df["model"].nunique())
print("Evaluations:", len(learning_models_df))

display(learning_model_stats.round(4))

Models included: 8
Evaluations: 40


test_accuracy                              \
                            min       max     mean   median   
feature_set                                                   
M1 - Lexical            84.9683   99.7408  97.8789  99.7366   
M2 - Derived            87.0119   99.7366  98.0324  99.6899   
M3 - URL Features       99.0143   99.7323  99.6017  99.6835   
M4 - Page-aware         99.9193  100.0000  99.9623  99.9575   
M5 - All numeric        99.9830  100.0000  99.9942  99.9958   

                  test_balanced_accuracy                              \
                                     min       max     mean   median   
feature_set                                                            
M1 - Lexical                     83.2550   99.7054  97.6316  99.6922   
M2 - Derived                     85.3867   99.6954  97.8032  99.6565   
M3 - URL Features                98.9638   99.6904  99.5623  99.6472   
M4 - Page-aware                  99.9156  100.0000  99.9598  99.9547   
M5 - All numeric                 99.9801  100.0000  99.9932  99.9950   

                  test_phishing_recall                              \
                                   min       max     mean   median   
feature_set                                                          
M1 - Lexical                   71.5082   99.4727  95.9361  99.4131   
M2 - Derived                   74.2439   99.4329  96.2321  99.4031   
M3 - URL Features              98.6172   99.4329  99.2924  99.3733   
M4 - Page-aware                99.8906  100.0000  99.9428  99.9353   
M5 - All numeric               99.9602  100.0000  99.9863  99.9901   

                  test_phishing_precision                             
                                      min    max      mean    median  
feature_set                                                           
M1 - Lexical                      91.4271  100.0   98.8634   99.9201  
M2 - Derived                      94.0991  100.0   99.0052   99.8403  
M3 - URL Features                 99.0706  100.0   99.7740   99.8601  
M4 - Page-aware                   99.9204  100.0   99.9689   99.9652  
M5 - All numeric                 100.0000  100.0  100.0000  100.0000

### Representative Error Analysis:

In [23]:
import json

with open("../results/Final_Tests/final_45_test_results.json") as f:
    final = json.load(f)

m3 = {
    r["model"]: r
    for r in final
    if r["feature_set"] == "M3 - URL Features"
}

for name in [
    "Ridge Logistic Regression",
    "Linear SVM",
    "Decision Tree",
    "XGBoost"
]:
    print(name, ":", m3[name]["hyperparameters"])

Ridge Logistic Regression : {'penalty': 'l2', 'C': 1000.0, 'solver': 'lbfgs', 'max_iter': 1000}
Linear SVM : {'C': 1000.0, 'max_iter': 3000}
Decision Tree : {'max_depth': 20, 'min_samples_leaf': 1, 'criterion': 'entropy'}
XGBoost : {'n_estimators': 200, 'learning_rate': 0.2, 'max_depth': 7, 'subsample': 1.0}


In [26]:
from sklearn.metrics import confusion_matrix

feats = feature_columns["M3 - URL Features"]

model_names = [
    "Ridge Logistic Regression",
    "Linear SVM",
    "Decision Tree",
    "XGBoost"
]

fitted = {}

for name in model_names:

    print(f"\nTraining {name}...")

    model = create_final_model(m3[name])

    model.fit(
        X_train_val[feats],
        y_train_val
    )

    predictions = model.predict(X_test[feats])

    cm = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    )

    reproduced = (cm[0, 1], cm[1, 0])

    saved = (
        m3[name]["test_missed_phishing"],
        m3[name]["test_false_alarms"]
    )

    print("Reproduced:", reproduced)
    print("Original:  ", saved)
    print("Status:    ", "OK" if reproduced == saved else "MISMATCH")

    fitted[name] = model

models = {
    "XGBoost": fitted["XGBoost"],
    "Ridge": fitted["Ridge Logistic Regression"],
    "Linear SVM": fitted["Linear SVM"],
    "Decision Tree": fitted["Decision Tree"]
}


Training Ridge Logistic Regression...
Reproduced: (np.int64(63), np.int64(6))
Original:   (63, 6)
Status:     OK

Training Linear SVM...
Reproduced: (np.int64(64), np.int64(0))
Original:   (64, 0)
Status:     OK

Training Decision Tree...
Reproduced: (np.int64(65), np.int64(27))
Original:   (65, 27)
Status:     OK

Training XGBoost...
Reproduced: (np.int64(60), np.int64(3))
Original:   (60, 3)
Status:     OK


#### Finding how many misses happened: 

In [27]:
feats = feature_columns["M3 - URL Features"]

phish = X_test.index[y_test.values == 0]
legit = X_test.index[y_test.values == 1]

missed = {
    n: set(phish[m.predict(X_test.loc[phish, feats]) == 1])
    for n, m in models.items()
}

alarms = {
    n: set(legit[m.predict(X_test.loc[legit, feats]) == 0])
    for n, m in models.items()
}

common = set.intersection(*missed.values())
anyone = set.union(*missed.values())
caught = set(phish) - anyone

print({n: len(s) for n, s in missed.items()})
print("Missed by all four:", len(common))
print("Missed by at least one:", len(anyone))
print("Caught by all four:", len(caught))

{'XGBoost': 60, 'Ridge': 63, 'Linear SVM': 64, 'Decision Tree': 65}
Missed by all four: 39
Missed by at least one: 89
Caught by all four: 9963


### Feature profile comparision & HTTP Analysis:

#### This basically answers: What makes those 39 consistently missed phishing URLs different from phishing URLs that our models successfully detected?

In [28]:
cols = [
    "URLLength",
    "NoOfSubDomain",
    "NoOfDegitsInURL",
    "NoOfOtherSpecialCharsInURL"
]

print(pd.concat({
    "missed by all": df.loc[list(common), cols].median(),
    "caught by all": df.loc[list(caught), cols].median(),
    "legitimate": df.loc[list(legit), cols].median()
}, axis=1))

print("Share using HTTPS:", {
    k: round(df.loc[list(v), "IsHTTPS"].mean(), 2)
    for k, v in {
        "missed by all": common,
        "caught by all": caught,
        "legitimate": set(legit)
    }.items()
})

                            missed by all  caught by all  legitimate
URLLength                            27.0           34.0        26.0
NoOfSubDomain                         1.0            1.0         1.0
NoOfDegitsInURL                       0.0            0.0         0.0
NoOfOtherSpecialCharsInURL            1.0            3.0         1.0
Share using HTTPS: {'missed by all': np.float64(1.0), 'caught by all': np.float64(0.49), 'legitimate': np.float64(1.0)}


#### Real URLs that got missed:

In [29]:
def defang(u):
    return u.replace("http", "hxxp").replace(".", "[.]")

print("PHISHING URLS MISSED BY ALL FOUR MODELS:\n")

for u in df.loc[list(common)[:8], "URL"]:
    print(defang(u))

print("\nFALSE ALARMS:\n")

for u in df.loc[list(set.union(*alarms.values()))[:5], "URL"]:
    print("False alarm:", defang(u))

PHISHING URLS MISSED BY ALL FOUR MODELS:

hxxps://www[.]k365v[.]com
hxxps://www[.]amznn[.]net
hxxps://www[.]coinbesaprime-logi[.]com
hxxps://www[.]eventsreg[.]in
hxxps://www[.]tallpullmoby[.]com
hxxps://www[.]serviziofatturazionearuba[.]com
hxxps://www[.]shorttermtradings[.]com
hxxps://www[.]divmainbot[.]pages[.]dev

FALSE ALARMS:

False alarm: hxxps://www[.]philippine-embassy[.]org[.]sg
False alarm: hxxps://www[.]environmentalhistory-au-nz[.]org
False alarm: hxxps://www[.]association-of-noise-consultants[.]co[.]uk
False alarm: hxxps://www[.]greenify-me[.]com
False alarm: hxxps://www[.]driftmasters[.]gp


<hr>